## Problema 5: Detección de Anomalías en Producción

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

In [ ]:
# CARGA DE DATOS
df_produccion = pd.read_csv('datos/produccion_agricola.csv')
df_climaticas = pd.read_csv('datos/condiciones_climaticas.csv')

In [ ]:
# Mapear los nulos en la columna 'eventos_extremos' a "sin evento"
df_climaticas['eventos_extremos'] = df_climaticas['eventos_extremos'].fillna('sin evento')

In [ ]:
# Excluimos la caña de azúcar para evitar el sesgo de magnitud
df_produccion = df_produccion[df_produccion['cultivo'] != 'Caña de azúcar']

In [ ]:
# UNIÓN DE LOS DATASETS
df_merged = pd.merge(df_produccion, df_climaticas, on=['anio', 'pais'], how='inner')

# Selección de variables para el análisis multivariante
features = [
    'rendimiento_ton_ha', 
    'superficie_hectareas', 
    'temperatura_promedio', 
    'precipitacion_total'
]

In [ ]:
# Limpieza: eliminamos filas con valores nulos en las columnas seleccionadas
df_anomalias = df_merged.dropna(subset=features).copy()

In [ ]:
# PREPROCESAMIENTO
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_anomalias[features])

In [ ]:
# ENTRENAMIENTO DE MODELOS

# --- Modelo A: Isolation Forest ---
# contamination=0.05 indica que esperamos un 5% de anomalías
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df_anomalias['anomaly_score_iso'] = iso_forest.fit(X_scaled).decision_function(X_scaled)

# --- Modelo B: One-Class SVM ---
# nu=0.05 es el equivalente a la tasa de anomalías esperada
oc_svm = OneClassSVM(nu=0.05, kernel="rbf", gamma='auto')
df_anomalias['anomaly_score_svm'] = oc_svm.fit(X_scaled).decision_function(X_scaled)

In [ ]:
# Percentiles
y_low = df_anomalias['rendimiento_ton_ha'].quantile(0.10)
y_high = df_anomalias['rendimiento_ton_ha'].quantile(0.90)

p_low = df_anomalias['precipitacion_total'].quantile(0.10)
p_high = df_anomalias['precipitacion_total'].quantile(0.90)

# Definir anomalías
df_anomalias['anomalia_real'] = (
    (df_anomalias['rendimiento_ton_ha'] < y_low) |
    (df_anomalias['rendimiento_ton_ha'] > y_high) |
    (df_anomalias['precipitacion_total'] < p_low) |
    (df_anomalias['precipitacion_total'] > p_high)
).astype(int)

def calcular_precision_at_k(df, score_col, target_col, k):
    # Los scores más bajos (negativos) son los más anómalos
    top_k = df.sort_values(by=score_col).head(k)
    precision = top_k[target_col].sum() / k
    return precision

# Calculamos K
k_valor = int(len(df_anomalias) * 0.05)

precision_iso = calcular_precision_at_k(df_anomalias, 'anomaly_score_iso', 'anomalia_real', k_valor)
precision_svm = calcular_precision_at_k(df_anomalias, 'anomaly_score_svm', 'anomalia_real', k_valor)

In [ ]:
print(" DETECCIÓN DE ANOMALÍAS\n")
print(f"Registros: {len(df_anomalias)}")
print(f"K: {k_valor}")
print(f"Precision@{k_valor} - Isolation Forest: {precision_iso:.4f}")
print(f"Precision@{k_valor} - One-Class SVM:    {precision_svm:.4f}")

 DETECCIÓN DE ANOMALÍAS

Registros: 1856
K: 92
Precision@92 - Isolation Forest: 0.7283
Precision@92 - One-Class SVM:    0.8370


In [ ]:
print("\nTOP 5 ANOMALÍAS ISO:")
print(df_anomalias.sort_values('anomaly_score_iso').head(5)[
    ['anio', 'pais', 'cultivo', 'rendimiento_ton_ha', 'precipitacion_total']
])

print("\nTOP 5 ANOMALÍAS SVM:")
print(df_anomalias.sort_values('anomaly_score_svm').head(5)[
    ['anio', 'pais', 'cultivo', 'rendimiento_ton_ha', 'precipitacion_total']
])


TOP 5 ANOMALÍAS ISO:
      anio            pais cultivo  rendimiento_ton_ha  precipitacion_total
1378  2019        Alemania    Maíz               11.03                  626
1304  2020         Francia    Maíz                8.35                 1387
690   2015  Estados Unidos    Maíz               11.00                  669
1255  2015         Francia    Maíz                6.75                 1455
1636  2020       Australia    Maíz                7.04                 1620

TOP 5 ANOMALÍAS SVM:
      anio            pais cultivo  rendimiento_ton_ha  precipitacion_total
1378  2019        Alemania    Maíz               11.03                  626
1529  2016       Australia   Trigo                7.60                  290
1255  2015         Francia    Maíz                6.75                 1455
576   2011  Estados Unidos    Maíz               10.04                 1531
1535  2016       Australia    Maíz                6.08                  290
